# NB02 — Macro Regime & Geopolitical Context

**Objectives**:
- Map the 10-year window onto macro regimes: QE era, rate hikes, COVID, zero-rate recovery, inflation shock, AI boom, tariff volatility
- Construct composite macro factor via PCA: yield curve slope + VIX + DXY + CPI surprise → 2-3 principal components
- Event study methodology: abnormal returns around 9 key events
- Cross-asset correlation analysis: tech stocks vs TLT, GLD, DXY under each regime
- Granger causality tests: Do macro factors predict tech returns?
- Structural break detection on rolling betas
- Regime-conditional summary statistics

**Dependencies**: NB01 (`master_data.parquet`)

**Output**: `macro_regimes.parquet`

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.stattools import grangercausalitytests
from src.config import *
from src.data_loader import download_fred
from src.feature_engineering import compute_log_returns, compute_simple_returns, rolling_beta, rolling_correlation
from src.visualization import save_fig, plot_correlation_heatmap
import logging
logging.basicConfig(level=logging.INFO)
print('Imports OK')

## 1. Load Master Data & FRED Macro

In [ ]:
master = pd.read_parquet(MASTER_DATA_FILE)
adj_tickers = [t for t in TICKERS if t in master.columns]
print(f'Master: {master.shape}')

# Identify benchmark columns (prefixed with BM_)
bm_cols = [c for c in master.columns if c.startswith('BM_')]
print(f'Benchmark columns: {bm_cols}')

# FRED macro data
fred_path = RAW_DIR / 'fred_macro.csv'
if fred_path.exists():
    macro = pd.read_csv(fred_path, index_col=0, parse_dates=True)
    print(f'FRED macro: {macro.shape}')
else:
    print('FRED data not available — using VIX + benchmarks as macro proxies')
    macro = pd.DataFrame()

# Compute returns for tickers and benchmarks
log_ret = compute_log_returns(master[adj_tickers])
simple_ret = compute_simple_returns(master[adj_tickers])

## 2. Define Macro Regimes

In [ ]:
# Rule-based macro regime labels (from known historical periods)
def label_regime(date):
    if date < pd.Timestamp('2018-01-01'): return 'QE Era'
    elif date < pd.Timestamp('2019-01-01'): return 'Rate Hike'
    elif date < pd.Timestamp('2020-02-19'): return 'Late Cycle'
    elif date < pd.Timestamp('2020-04-01'): return 'COVID Crash'
    elif date < pd.Timestamp('2021-12-01'): return 'Zero-Rate Recovery'
    elif date < pd.Timestamp('2023-10-01'): return 'Inflation/Rate Shock'
    elif date < pd.Timestamp('2025-01-01'): return 'AI Boom'
    else: return 'Tariff/Geopolitical'

regime_labels = pd.Series(master.index.map(label_regime), index=master.index, name='macro_regime')

# Regime counts and date ranges
print('--- Macro Regime Timeline ---')
for regime in ['QE Era', 'Rate Hike', 'Late Cycle', 'COVID Crash',
               'Zero-Rate Recovery', 'Inflation/Rate Shock', 'AI Boom', 'Tariff/Geopolitical']:
    mask = regime_labels == regime
    if mask.any():
        dates = master.index[mask]
        n_days = mask.sum()
        print(f'  {regime:25s}: {dates[0].date()} → {dates[-1].date()} ({n_days} days)')

# Visualize regime timeline
fig, ax = plt.subplots(figsize=(16, 3))
regime_colors = {
    'QE Era': 'lightblue', 'Rate Hike': 'orange', 'Late Cycle': 'lightyellow',
    'COVID Crash': 'red', 'Zero-Rate Recovery': 'lightgreen',
    'Inflation/Rate Shock': 'salmon', 'AI Boom': 'gold', 'Tariff/Geopolitical': 'plum'
}
for regime, color in regime_colors.items():
    mask = regime_labels == regime
    if mask.any():
        dates = master.index[mask]
        ax.axvspan(dates[0], dates[-1], alpha=0.6, color=color, label=regime)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.set_title('Macro Regime Timeline (2016-2026)')
ax.set_yticks([])
fig.tight_layout()
save_fig(fig, 'nb02_regime_timeline')
plt.show()

## 3. Event Study: Abnormal Returns

In [ ]:
# Event study: cumulative abnormal returns around key events
event_results = []
for event_name, (start, end) in KEY_EVENTS.items():
    mask = (log_ret.index >= start) & (log_ret.index <= end)
    event_ret = log_ret.loc[mask]
    if len(event_ret) == 0:
        continue
    cum_ret = event_ret.sum()  # sum of log returns ≈ log cumulative return
    for t in adj_tickers:
        event_results.append({
            'event': event_name, 'ticker': t,
            'cumulative_return': cum_ret.get(t, np.nan),
            'n_days': len(event_ret)
        })

event_df = pd.DataFrame(event_results)
event_pivot = event_df.pivot(index='ticker', columns='event', values='cumulative_return')

# Heatmap of event returns
fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(event_pivot, annot=True, fmt='.1%', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Cumulative Log Returns During Key Events')
fig.tight_layout()
save_fig(fig, 'nb02_event_returns_heatmap')
plt.show()

# Worst hit tickers per event
print('\n--- Most Affected Tickers Per Event ---')
for event in event_pivot.columns:
    worst = event_pivot[event].idxmin()
    best = event_pivot[event].idxmax()
    print(f'  {event:30s}: Worst={worst} ({event_pivot[event].min():.1%}), '
          f'Best={best} ({event_pivot[event].max():.1%})')

## 4. Granger Causality

In [ ]:
# Granger causality: VIX → tech sector returns
# Build macro factor variables from benchmarks
vix_col = [c for c in master.columns if 'VIX' in c and 'VVIX' not in c]
tlt_col = [c for c in master.columns if 'TLT' in c]
dxy_col = [c for c in master.columns if 'DX-Y' in c]
xlk_col = [c for c in master.columns if 'XLK' in c]

macro_factors = pd.DataFrame(index=master.index)
if vix_col:
    macro_factors['vix_change'] = master[vix_col[0]].pct_change()
if tlt_col:
    macro_factors['tlt_return'] = compute_log_returns(master[[tlt_col[0]]])[tlt_col[0]]
if dxy_col:
    macro_factors['dxy_change'] = master[dxy_col[0]].pct_change()
if xlk_col:
    macro_factors['xlk_return'] = compute_log_returns(master[[xlk_col[0]]])[xlk_col[0]]

# Equal-weight tech portfolio return as proxy
ew_tech = log_ret[adj_tickers].mean(axis=1)
macro_factors['ew_tech_return'] = ew_tech

macro_factors = macro_factors.dropna()
print(f'Macro factors: {macro_factors.shape}')

# Run Granger causality tests
granger_results = []
# Run Granger causality for each factor independently
for factor in ['vix_change', 'tlt_return', 'dxy_change']:
    if factor not in macro_factors.columns:
        continue
    test_data = macro_factors[['ew_tech_return', factor]].dropna()
    if len(test_data) < 100:
        continue
    print(f'\n--- Granger Causality: {factor} → EW Tech Return ---')
    try:
        gc = grangercausalitytests(test_data[['ew_tech_return', factor]], maxlag=5, verbose=False)
        for lag in range(1, 6):
            f_stat = gc[lag][0]['ssr_ftest'][0]
            p_val = gc[lag][0]['ssr_ftest'][1]
            sig = '*' if p_val < 0.05 else ''
            granger_results.append({
                'factor': factor, 'lag': lag, 'f_stat': f_stat, 'p_value': p_val
            })
            print(f'  Lag {lag}: F={f_stat:.3f}, p={p_val:.4f} {sig}')
    except Exception as e:
        print(f'  Failed: {e}')

granger_df = pd.DataFrame(granger_results)
if not granger_df.empty:
    print(f'\nSignificant Granger relationships (p<0.05): '
          f'{(granger_df["p_value"] < 0.05).sum()} / {len(granger_df)}')

## 4b. BH-FDR Correction on Granger Causality Tests

Apply Benjamini-Hochberg FDR correction to Granger p-values to control for
multiple testing across factor-lag combinations.

In [ ]:
from src.statistical_tests import benjamini_hochberg

if not granger_df.empty:
    raw_pvals = granger_df['p_value'].values
    rejected, adjusted = benjamini_hochberg(raw_pvals, q=0.05)
    granger_df['p_value_bh'] = adjusted
    granger_df['reject_bh'] = rejected
    print("--- BH-FDR Corrected Granger Causality ---")
    print(f"Significant after BH correction: {sum(rejected)}/{len(rejected)}")
    print(granger_df[granger_df['reject_bh']][['factor', 'lag', 'f_stat', 'p_value', 'p_value_bh']])
else:
    print("No Granger results to correct")

## 4c. Engle-Granger Cointegration Tests (5 Natural Pairs)

Test for long-run equilibrium relationships between natural pairs.
Cointegrated pairs mean-revert, enabling pairs trading strategies.
Half-life measures speed of mean-reversion.

In [ ]:
from src.feature_engineering import engle_granger_cointegration

adj_close = master[adj_tickers]
pairs = [('NVDA', 'AMD'), ('TSM', 'AVGO'), ('META', 'GOOG'), ('PANW', 'CRWD'), ('CRM', 'NOW')]

coint_results = {}
print("--- Engle-Granger Cointegration Tests ---")
print(f"{'Pair':>15s}  {'Beta':>7s}  {'ADF Stat':>9s}  {'p-value':>8s}  {'Half-life':>10s}  {'Coint?':>6s}")
print("-" * 65)

for a, b in pairs:
    if a in adj_close.columns and b in adj_close.columns:
        result = engle_granger_cointegration(adj_close[a], adj_close[b])
        coint_results[f'{a}/{b}'] = result
        coint = 'Yes' if result['p_value'] < 0.05 else 'No'
        print(f"{a+'/'+b:>15s}  {result['beta']:7.3f}  {result['adf_stat']:9.3f}  "
              f"{result['p_value']:8.4f}  {result['half_life']:10.1f}d  {coint:>6s}")

# Save cointegration results
coint_df = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk != 'residuals'}
                          for k, v in coint_results.items()}).T
coint_df.to_csv(TABLES_DIR / 'nb02_cointegration_pairs.csv')
print(f"\nSaved to {TABLES_DIR / 'nb02_cointegration_pairs.csv'}")

## 5. Composite Macro Factor (PCA)

Reduce VIX change, TLT return, DXY change, and XLK return into 2-3 uncorrelated macro PCs.
These PCs are saved to  as features for NB07-08.


In [ ]:
# Composite macro factor via PCA
# macro_factors already built in Cell 9 (vix_change, tlt_return, dxy_change, xlk_return)
pca_feature_cols = [c for c in ["vix_change", "tlt_return", "dxy_change", "xlk_return"]
                    if c in macro_factors.columns]
pca_data = macro_factors[pca_feature_cols].dropna()

scaler = StandardScaler()
pca_scaled = scaler.fit_transform(pca_data)

pca = PCA(n_components=min(3, len(pca_feature_cols)))
pca_components = pca.fit_transform(pca_scaled)

pca_df = pd.DataFrame(
    pca_components,
    index=pca_data.index,
    columns=[f"macro_pc{i+1}" for i in range(pca.n_components_)]
)

print(f"PCA macro factors: {pca_df.shape}")
print(f"Explained variance: "
      + ", ".join(f"PC{i+1}={v:.1%}" for i, v in enumerate(pca.explained_variance_ratio_)))
print(f"Total: {pca.explained_variance_ratio_.sum():.1%}")


## 6. Cross-Asset Correlation by Regime

How do tech stocks correlate with bonds (TLT), gold (GLD), and the dollar (DXY)
under different macro regimes? Correlation breakdown during crises is a key risk factor.

In [ ]:
# Cross-asset correlation: EW Tech vs TLT, GLD, DXY per regime
cross_asset_cols = {}
gld_col = [c for c in master.columns if 'GLD' in c]
for name, col_list in [('TLT', tlt_col), ('GLD', gld_col), ('DXY', dxy_col)]:
    if col_list:
        cross_asset_cols[name] = compute_log_returns(master[[col_list[0]]])[col_list[0]]

if cross_asset_cols:
    regimes_ordered = ['QE Era', 'Rate Hike', 'Late Cycle', 'COVID Crash',
                       'Zero-Rate Recovery', 'Inflation/Rate Shock', 'AI Boom', 'Tariff/Geopolitical']
    cross_corr_table = []
    for regime in regimes_ordered:
        mask = regime_labels == regime
        if mask.sum() < 30:
            continue
        for asset_name, asset_ret in cross_asset_cols.items():
            aligned = pd.DataFrame({'tech': ew_tech, 'asset': asset_ret}).loc[mask].dropna()
            if len(aligned) < 20:
                continue
            corr = aligned['tech'].corr(aligned['asset'])
            cross_corr_table.append({
                'regime': regime, 'asset': asset_name, 'correlation': corr,
                'n_days': len(aligned)
            })

    cross_df = pd.DataFrame(cross_corr_table)
    cross_pivot = cross_df.pivot(index='regime', columns='asset', values='correlation')
    cross_pivot = cross_pivot.reindex(regimes_ordered).dropna(how='all')

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(cross_pivot, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-0.5, vmax=0.5, linewidths=0.5, ax=ax)
    ax.set_title('EW Tech Portfolio Correlation with Cross-Asset Classes by Regime')
    fig.tight_layout()
    save_fig(fig, 'nb02_cross_asset_correlation')
    plt.show()
else:
    print('Cross-asset columns not available in master data')

## 7. Regime-Conditional Summary Statistics

Mean return, volatility, skewness, and kurtosis vary dramatically across regimes.
These tables inform the regime-adaptive portfolio strategy in NB11.

In [ ]:
# Regime-conditional statistics for all 20 tickers
regime_stats = []
for regime in regimes_ordered:
    mask = regime_labels == regime
    sub_ret = log_ret.loc[mask]
    for t in adj_tickers:
        s = sub_ret[t].dropna()
        if len(s) < 10:
            continue
        regime_stats.append({
            'regime': regime, 'ticker': t,
            'ann_return': s.mean() * 252,
            'ann_vol': s.std() * np.sqrt(252),
            'sharpe': (s.mean() * 252) / (s.std() * np.sqrt(252)) if s.std() > 0 else 0,
            'skewness': s.skew(),
            'excess_kurtosis': s.kurtosis(),
            'n_days': len(s)
        })

regime_stats_df = pd.DataFrame(regime_stats)

# Summary: average stats across tickers per regime
regime_summary = regime_stats_df.groupby('regime').agg({
    'ann_return': 'mean', 'ann_vol': 'mean', 'sharpe': 'mean',
    'skewness': 'mean', 'excess_kurtosis': 'mean', 'n_days': 'first'
}).reindex(regimes_ordered).dropna(how='all')

print('--- Average Regime Statistics (across 20 tickers) ---')
regime_summary.style.format({
    'ann_return': '{:.1%}', 'ann_vol': '{:.1%}', 'sharpe': '{:.2f}',
    'skewness': '{:.3f}', 'excess_kurtosis': '{:.2f}', 'n_days': '{:.0f}'
})

## 8. Structural Break Detection (CUSUM on Rolling Betas)

Detect structural breaks in the relationship between tech stocks and SPY.
Breaks in beta indicate regime changes in systematic risk exposure.

In [ ]:
# Rolling beta of EW tech portfolio vs SPY
spy_col = [c for c in master.columns if 'SPY' in c]
if spy_col:
    spy_ret = compute_log_returns(master[[spy_col[0]]])[spy_col[0]]

    # Rolling 63-day beta for selected tickers
    beta_tickers = ['NVDA', 'AAPL', 'MSFT', 'CRWD', 'PLTR', 'TSM']
    fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True)
    for ax, t in zip(axes.flatten(), beta_tickers):
        if t not in adj_tickers:
            continue
        beta = rolling_beta(log_ret[t], spy_ret, window=63)
        ax.plot(beta.index, beta.values, linewidth=0.8)
        ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Beta=1')
        ax.set_title(f'{t} — Rolling 63d Beta vs SPY')
        ax.set_ylabel('Beta')
        ax.legend(fontsize=8)
        # Shade key events
        for ev_name, (ev_start, ev_end) in KEY_EVENTS.items():
            ax.axvspan(ev_start, ev_end, alpha=0.1, color='orange')

    fig.suptitle('Rolling 63-Day Beta vs SPY (key events shaded)', fontsize=13)
    fig.tight_layout()
    save_fig(fig, 'nb02_rolling_betas')
    plt.show()

    # CUSUM-style analysis: cumulative sum of demeaned beta changes
    ew_beta = rolling_beta(ew_tech, spy_ret, window=63).dropna()
    beta_change = ew_beta.diff()
    cusum = (beta_change - beta_change.mean()).cumsum()

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    ax1.plot(ew_beta.index, ew_beta.values, linewidth=0.8)
    ax1.set_ylabel('EW Tech Beta vs SPY')
    ax1.set_title('Rolling 63-Day Beta: EW Tech Portfolio')
    ax2.plot(cusum.index, cusum.values, linewidth=0.8, color='darkred')
    ax2.axhline(0, color='black', linewidth=0.3)
    ax2.set_ylabel('CUSUM of Beta Changes')
    ax2.set_title('CUSUM — Structural Break Indicator (slope changes = breaks)')
    for ev_name, (ev_start, ev_end) in KEY_EVENTS.items():
        ax1.axvspan(ev_start, ev_end, alpha=0.1, color='orange')
        ax2.axvspan(ev_start, ev_end, alpha=0.1, color='orange')
    fig.tight_layout()
    save_fig(fig, 'nb02_cusum_beta')
    plt.show()
else:
    print('SPY not found in master data — skipping beta analysis')

## 9. Save Outputs


In [ ]:
# Save regime labels with PCA factors
regime_out = pd.DataFrame({'macro_regime': regime_labels}, index=master.index)
# Add PCA factors (aligned to master index)
for col in pca_df.columns:
    regime_out[col] = pca_df[col].reindex(master.index)
regime_out.to_parquet(MACRO_REGIMES_FILE)
print(f'Saved: {MACRO_REGIMES_FILE} ({regime_out.shape})')

# Save regime stats and event study results
regime_stats_df.to_csv(TABLES_DIR / 'nb02_regime_conditional_stats.csv', index=False)
event_pivot.to_csv(TABLES_DIR / 'nb02_event_study_returns.csv')
if not granger_df.empty:
    granger_df.to_csv(TABLES_DIR / 'nb02_granger_causality.csv', index=False)

print('\n--- NB02 Complete ---')
print(f'  Regimes: {regime_labels.nunique()} macro regimes labeled')
print(f'  PCA: {pca_df.shape[1]} principal components ({pca.explained_variance_ratio_.sum():.1%} total variance)')
print(f'  Events: {len(KEY_EVENTS)} key events analyzed')
print(f'\nReady for: NB05 (regime detection can use macro factors), NB07-08 (macro features for ML)')